In [2]:
import glob
import pandas as pd
import numpy as np

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

# ==========================================================
# 1) إيجاد مجلد الداتا تلقائياً
# ==========================================================
BASE = glob.glob("**/CMAPSSData", recursive=True)[0]
print("مجلد الداتا:", BASE)

# ==========================================================
# 2) تسمية الأعمدة الـ 26
#    الترتيب ثابت في كل ملفات C-MAPSS
# ==========================================================
SENSORS = [f"s{i}" for i in range(1, 22)]
COLS = ["unit", "cycle", "op1", "op2", "op3"] + SENSORS


def load(name):
    """قراءة ملف C-MAPSS: بدون هيدر، الفاصل مسافات متعددة."""
    return pd.read_csv(f"{BASE}/{name}", sep=r"\s+",
                       header=None, names=COLS)


train = load("train_FD001.txt")
test = load("test_FD001.txt")
rul_true = pd.read_csv(f"{BASE}/RUL_FD001.txt",
                       sep=r"\s+", header=None, names=["RUL"])

# ==========================================================
# 3) فحص أن القراءة صحيحة قبل أي شيء آخر
# ==========================================================
print("\nالأبعاد:")
print("train :", train.shape)   # المتوقع: (20631, 26)
print("test  :", test.shape)    # المتوقع: (13096, 26)
print("RUL   :", rul_true.shape)  # المتوقع: (100, 1)

print("\nهل يوجد قيم مفقودة؟", train.isna().sum().sum())

print("\nأول 3 صفوف:")
print(train.head(3))

مجلد الداتا: 6.+Turbofan+Engine+Degradation+Simulation+Data+Set\6. Turbofan Engine Degradation Simulation Data Set\CMAPSSData

الأبعاد:
train : (20631, 26)
test  : (13096, 26)
RUL   : (100, 1)

هل يوجد قيم مفقودة؟ 0

أول 3 صفوف:
   unit  cycle     op1     op2    op3      s1      s2       s3       s4     s5     s6      s7       s8       s9  s10    s11     s12      s13      s14     s15   s16  s17   s18    s19    s20      s21
0     1      1 -0.0007 -0.0004  100.0  518.67  641.82  1589.70  1400.60  14.62  21.61  554.36  2388.06  9046.19  1.3  47.47  521.66  2388.02  8138.62  8.4195  0.03  392  2388  100.0  39.06  23.4190
1     1      2  0.0019 -0.0003  100.0  518.67  642.15  1591.82  1403.14  14.62  21.61  553.75  2388.04  9044.07  1.3  47.49  522.28  2388.07  8131.49  8.4318  0.03  392  2388  100.0  39.00  23.4236
2     1      3 -0.0043  0.0003  100.0  518.67  642.35  1587.99  1404.20  14.62  21.61  554.26  2388.08  9052.94  1.3  47.27  522.42  2388.03  8133.23  8.4178  0.03  390  2388  1

In [3]:
# ==========================================================
# 4) بنية الداتا على مستوى المحرك
# ==========================================================
len_train = train.groupby("unit")["cycle"].max()
len_test = test.groupby("unit")["cycle"].max()

print("عدد محركات train:", train["unit"].nunique())
print("عدد محركات test :", test["unit"].nunique())

print("\nطول مسار المحرك في train:")
print(len_train.describe())

print("\nطول مسار المحرك في test:")
print(len_test.describe())

# ==========================================================
# 5) الانحراف المعياري لكل عمود
#    صفر = العمود ثابت طوال الداتا كلها
# ==========================================================
print("\n" + "=" * 55)
print("الانحراف المعياري (مرتب تصاعدياً)")
print("=" * 55)
stds = train[["op1", "op2", "op3"] + SENSORS].std().sort_values()
print(stds.to_string())

# ==========================================================
# 6) شكل التدهور عبر عمر المحرك رقم 1
#    عيّنة من 10 نقاط موزعة على كامل عمره
# ==========================================================
print("\n" + "=" * 55)
print("سلوك حساسات مختارة عبر عمر المحرك رقم 1")
print("=" * 55)
u1 = train[train["unit"] == 1]
idx = np.linspace(0, len(u1) - 1, 10).astype(int)
print(u1.iloc[idx][["cycle", "s2", "s4", "s7", "s11", "s12", "s21"]]
      .to_string(index=False))

# ==========================================================
# 7) العلاقة بين test وملف RUL
# ==========================================================
print("\n" + "=" * 55)
print("آخر دورة في test مقابل القيمة الحقيقية")
print("=" * 55)
print(pd.DataFrame({
    "unit": len_test.index[:10],
    "last_cycle_in_test": len_test.values[:10],
    "true_RUL": rul_true["RUL"].values[:10],
}).to_string(index=False))

عدد محركات train: 100
عدد محركات test : 100

طول مسار المحرك في train:
count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: cycle, dtype: float64

طول مسار المحرك في test:
count    100.000000
mean     130.960000
std       53.593479
min       31.000000
25%       88.750000
50%      133.500000
75%      164.250000
max      303.000000
Name: cycle, dtype: float64

الانحراف المعياري (مرتب تصاعدياً)
op3    0.000000e+00
s19    0.000000e+00
s18    0.000000e+00
s16    1.556432e-14
s10    4.660829e-13
s5     3.394700e-12
s1     6.537152e-11
op2    2.930621e-04
s6     1.388985e-03
op1    2.187313e-03
s15    3.750504e-02
s8     7.098548e-02
s13    7.191892e-02
s21    1.082509e-01
s20    1.807464e-01
s11    2.670874e-01
s2     5.000533e-01
s12    7.375534e-01
s7     8.850923e-01
s17    1.548763e+00
s3     6.131150e+00
s4     9.000605e+00
s14    1.907618e+01
s9     2.208288e+01

سلوك حساسات

In [4]:
print("=" * 55)
print("الانحراف المعياري (مرتب تصاعدياً)")
print("=" * 55)
print(train[["op1", "op2", "op3"] + SENSORS].std().sort_values().to_string())

الانحراف المعياري (مرتب تصاعدياً)
op3    0.000000e+00
s19    0.000000e+00
s18    0.000000e+00
s16    1.556432e-14
s10    4.660829e-13
s5     3.394700e-12
s1     6.537152e-11
op2    2.930621e-04
s6     1.388985e-03
op1    2.187313e-03
s15    3.750504e-02
s8     7.098548e-02
s13    7.191892e-02
s21    1.082509e-01
s20    1.807464e-01
s11    2.670874e-01
s2     5.000533e-01
s12    7.375534e-01
s7     8.850923e-01
s17    1.548763e+00
s3     6.131150e+00
s4     9.000605e+00
s14    1.907618e+01
s9     2.208288e+01


In [5]:
u1 = train[train["unit"] == 1]
idx = np.linspace(0, len(u1) - 1, 10).astype(int)
print(u1.iloc[idx][["cycle", "s2", "s4", "s7", "s11", "s12", "s21"]]
      .to_string(index=False))

 cycle     s2      s4     s7   s11    s12     s21
     1 641.82 1400.60 554.36 47.47 521.66 23.4190
    22 642.77 1400.57 553.63 47.24 522.00 23.3792
    43 642.24 1406.05 553.68 47.32 522.74 23.3148
    64 642.23 1400.11 553.88 47.45 521.82 23.3840
    85 642.28 1405.96 554.40 47.36 522.30 23.2840
   107 642.53 1407.73 553.74 47.35 521.75 23.4086
   128 642.47 1407.06 552.74 47.62 521.96 23.2641
   149 642.90 1410.70 552.54 47.83 521.13 23.2565
   170 642.92 1417.41 552.02 47.86 520.23 23.1615
   192 643.54 1427.20 551.25 48.25 520.08 22.9649


In [6]:
# ==========================================================
# بناء عمود الهدف RUL على صفوف الـ train
# ==========================================================

# أقصى دورة لكل محرك = لحظة عطله
max_cycle = train.groupby("unit")["cycle"].transform("max")

# لكل صف: كم دورة باقية حتى العطل
train["RUL"] = max_cycle - train["cycle"]

# فحص النتيجة على المحرك رقم 1
print(train[train["unit"] == 1][["unit", "cycle", "RUL"]].head(3))
print("...")
print(train[train["unit"] == 1][["unit", "cycle", "RUL"]].tail(3))

print("\nتوزيع الهدف:")
print(train["RUL"].describe())

   unit  cycle  RUL
0     1      1  191
1     1      2  190
2     1      3  189
...
     unit  cycle  RUL
189     1    190    2
190     1    191    1
191     1    192    0

توزيع الهدف:
count    20631.000000
mean       107.807862
std         68.880990
min          0.000000
25%         51.000000
50%        103.000000
75%        155.000000
max        361.000000
Name: RUL, dtype: float64


In [7]:
print("الانحراف المعياري (مرتب تصاعدياً)")
print("=" * 45)
print(train[["op1", "op2", "op3"] + SENSORS].std().sort_values().to_string())

الانحراف المعياري (مرتب تصاعدياً)
op3    0.000000e+00
s19    0.000000e+00
s18    0.000000e+00
s16    1.556432e-14
s10    4.660829e-13
s5     3.394700e-12
s1     6.537152e-11
op2    2.930621e-04
s6     1.388985e-03
op1    2.187313e-03
s15    3.750504e-02
s8     7.098548e-02
s13    7.191892e-02
s21    1.082509e-01
s20    1.807464e-01
s11    2.670874e-01
s2     5.000533e-01
s12    7.375534e-01
s7     8.850923e-01
s17    1.548763e+00
s3     6.131150e+00
s4     9.000605e+00
s14    1.907618e+01
s9     2.208288e+01


In [8]:
u1 = train[train["unit"] == 1]
idx = np.linspace(0, len(u1) - 1, 8).astype(int)
print(u1.iloc[idx][["cycle", "RUL", "s2", "s4", "s7", "s11", "s12", "s21"]]
      .to_string(index=False))

 cycle  RUL     s2      s4     s7   s11    s12     s21
     1  191 641.82 1400.60 554.36 47.47 521.66 23.4190
    28  164 642.35 1399.13 554.68 47.41 522.57 23.2841
    55  137 642.41 1401.58 554.45 47.40 522.11 23.3567
    82  110 642.04 1399.17 554.18 47.25 521.95 23.2713
   110   82 642.55 1401.27 554.08 47.55 522.01 23.2951
   137   55 642.43 1405.15 553.39 47.89 521.63 23.2593
   164   28 643.17 1416.76 551.90 47.70 520.63 23.1761
   192    0 643.54 1427.20 551.25 48.25 520.08 22.9649


In [9]:
lengths = train.groupby("unit")["cycle"].max()
shortest = lengths.idxmin()   # أقصر محرك
longest = lengths.idxmax()    # أطول محرك

print(f"المحرك {shortest} - عمره {lengths[shortest]} دورة")
print(train[(train.unit == shortest) & (train.cycle <= 3)]
      [["cycle", "RUL", "s2", "s4", "s7", "s11", "s12"]].to_string(index=False))

print(f"\nالمحرك {longest} - عمره {lengths[longest]} دورة")
print(train[(train.unit == longest) & (train.cycle <= 3)]
      [["cycle", "RUL", "s2", "s4", "s7", "s11", "s12"]].to_string(index=False))

المحرك 39 - عمره 128 دورة
 cycle  RUL     s2      s4     s7   s11    s12
     1  127 642.72 1410.00 552.82 47.64 521.78
     2  126 642.61 1407.99 553.25 47.60 521.56
     3  125 642.26 1407.48 552.90 47.56 522.15

المحرك 69 - عمره 362 دورة
 cycle  RUL     s2      s4     s7   s11    s12
     1  361 641.96 1400.65 553.68 47.34 522.03
     2  360 641.50 1401.54 554.22 47.26 522.08
     3  359 642.73 1405.22 554.11 47.35 522.05


In [10]:
# ==========================================================
# متى تبدأ الحساسات فعلياً بالتحرك؟
# نقيس s4 كممثل، بعد توحيده لكل محرك على حدة
# (لأن كل محرك يبدأ من خط أساس مختلف قليلاً)
# ==========================================================

df = train.copy()

# خط الأساس لكل محرك = متوسط أول 20 دورة من عمره
baseline = (df[df.cycle <= 20]
            .groupby("unit")["s4"]
            .mean()
            .rename("base"))

df = df.merge(baseline, on="unit")

# الانحراف عن خط الأساس
df["dev"] = df["s4"] - df["base"]

# متوسط الانحراف عند كل قيمة RUL عبر كل المحركات
curve = df.groupby("RUL")["dev"].mean()

# نعرض من RUL=200 نزولاً حتى 0، كل 10 دورات
print("RUL  |  متوسط انحراف s4 عن خط الأساس")
print("-" * 42)
for r in range(200, -1, -10):
    if r in curve.index:
        print(f"{r:4d} |  {curve[r]:+8.3f}")

RUL  |  متوسط انحراف s4 عن خط الأساس
------------------------------------------
 200 |    -0.025
 190 |    +0.482
 180 |    +1.457
 170 |    +1.068
 160 |    +0.123
 150 |    +1.090
 140 |    +1.221
 130 |    +0.750
 120 |    +1.555
 110 |    +3.034
 100 |    +3.976
  90 |    +3.983
  80 |    +5.967
  70 |    +6.250
  60 |    +7.756
  50 |    +9.952
  40 |   +12.467
  30 |   +15.638
  20 |   +17.819
  10 |   +21.433
   0 |   +28.115


In [12]:
CAP = 120

train["RUL_capped"] = train["RUL"].clip(upper=CAP)

print("قبل السقف:")
print(train["RUL"].describe()[["mean", "std", "max"]])

print("\nبعد السقف:")
print(train["RUL_capped"].describe()[["mean", "std", "max"]])

# كم صف تأثر؟
n = (train["RUL"] > CAP).sum()
print(f"\nصفوف تغيرت: {n} من {len(train)}"
      f"  ({n/len(train)*100:.1f}%)")

# فحص الأثر على المحركين اللذين قارناهما
for u in [39, 69]:
    row = train[(train.unit == u) & (train.cycle == 1)]
    print(f"محرك {u}: RUL أصلي {row['RUL'].iloc[0]}"
          f"  →  بعد السقف {row['RUL_capped'].iloc[0]}")

قبل السقف:
mean    107.807862
std      68.880990
max     361.000000
Name: RUL, dtype: float64

بعد السقف:
mean     84.810237
std      39.858105
max     120.000000
Name: RUL_capped, dtype: float64

صفوف تغيرت: 8531 من 20631  (41.4%)
محرك 39: RUL أصلي 127  →  بعد السقف 120
محرك 69: RUL أصلي 361  →  بعد السقف 120


In [13]:
from sklearn.model_selection import train_test_split

# ==========================================================
# التقسيم على مستوى المحرك، لا على مستوى الصف
# ==========================================================

units = train["unit"].unique()          # 100 محرك

tr_units, va_units = train_test_split(
    units, test_size=0.2, random_state=42
)

tr = train[train["unit"].isin(tr_units)].copy()
va = train[train["unit"].isin(va_units)].copy()

print(f"محركات التدريب : {len(tr_units)}   صفوف: {len(tr)}")
print(f"محركات التحقق  : {len(va_units)}   صفوف: {len(va)}")

# فحص أساسي: لا يوجد محرك مشترك بين المجموعتين
overlap = set(tr["unit"]) & set(va["unit"])
print("محركات مشتركة:", len(overlap))

محركات التدريب : 80   صفوف: 16561
محركات التحقق  : 20   صفوف: 4070
محركات مشتركة: 0


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# الأعمدة الميتة التي استبعدناها سابقاً
dead = ["op3", "s19", "s18", "s16", "s10", "s5", "s1"]
FEATURES = [c for c in ["op1", "op2"] + SENSORS if c not in dead]

X_tr, y_tr = tr[FEATURES], tr["RUL_capped"]
X_va, y_va = va[FEATURES], va["RUL_capped"]

model = LinearRegression()
model.fit(X_tr, y_tr)

pred = model.predict(X_va)
rmse = mean_squared_error(y_va, pred) ** 0.5

print("عدد الأعمدة المستخدمة:", len(FEATURES))
print("RMSE على التحقق:", round(rmse, 2))

عدد الأعمدة المستخدمة: 17
RMSE على التحقق: 19.39


In [15]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_tr, y_tr)

rmse_dummy = mean_squared_error(y_va, dummy.predict(X_va)) ** 0.5

print("RMSE - تنبؤ ثابت :", round(rmse_dummy, 2))
print("RMSE - خطي       :", round(rmse, 2))
print("التحسّن           :", 
      f"{(1 - rmse/rmse_dummy)*100:.1f}%")

RMSE - تنبؤ ثابت : 39.92
RMSE - خطي       : 19.39
التحسّن           : 51.4%


In [16]:
def add_deviation(df, baseline_window=20):
    """
    يضيف لكل حساس عموداً يمثل انحرافه عن خط أساس المحرك نفسه.
    خط الأساس = متوسط أول عدة دورات من عمر المحرك.
    """
    df = df.copy()
    
    base = (df[df.cycle <= baseline_window]
            .groupby("unit")[FEATURES]
            .mean())
    
    for c in FEATURES:
        df[c + "_dev"] = df[c] - df["unit"].map(base[c])
    
    return df


tr2 = add_deviation(tr)
va2 = add_deviation(va)

DEV = [c + "_dev" for c in FEATURES]

print("أعمدة جديدة:", len(DEV))
print(tr2[["unit", "cycle", "s4", "s4_dev", "RUL_capped"]].head(3))

أعمدة جديدة: 17
     unit  cycle       s4  s4_dev  RUL_capped
192     2      1  1391.28  -6.073         120
193     2      2  1393.13  -4.223         120
194     2      3  1398.96   1.607         120


In [17]:
# التدريب على القراءات الأصلية + الانحرافات
COLS_2 = FEATURES + DEV

m2 = LinearRegression()
m2.fit(tr2[COLS_2], tr2["RUL_capped"])

rmse2 = mean_squared_error(
    va2["RUL_capped"], m2.predict(va2[COLS_2])
) ** 0.5

print("الأساس  (قراءات فقط)      :", round(rmse, 2))
print("مع الانحرافات             :", round(rmse2, 2))

الأساس  (قراءات فقط)      : 19.39
مع الانحرافات             : 16.09


In [18]:
WIN = 15   # طول النافذة بالدورات


def add_rolling(df, win=WIN):
    """
    لكل عمود انحراف، يضيف:
      _mean : متوسط النافذة  -> الموقع بلا ضجيج
      _slope: ميل النافذة    -> اتجاه التغير
    النافذة تحسب لكل محرك على حدة، وتنظر للماضي فقط.
    """
    df = df.sort_values(["unit", "cycle"]).copy()
    g = df.groupby("unit")

    for c in DEV:
        roll = g[c].rolling(win, min_periods=1)
        df[c + "_mean"] = roll.mean().values
        # الميل = تغير القيمة عبر النافذة مقسوماً على طولها
        df[c + "_slope"] = (
            (df[c] - g[c].shift(win - 1)) / (win - 1)
        ).fillna(0).values

    return df


tr3 = add_rolling(tr2)
va3 = add_rolling(va2)

ROLL = [c + s for c in DEV for s in ["_mean", "_slope"]]
print("أعمدة جديدة:", len(ROLL))
print(tr3[["unit", "cycle", "s4_dev",
           "s4_dev_mean", "s4_dev_slope"]].head(20).to_string(index=False))

أعمدة جديدة: 34
 unit  cycle  s4_dev  s4_dev_mean  s4_dev_slope
    2      1  -6.073    -6.073000      0.000000
    2      2  -4.223    -5.148000      0.000000
    2      3   1.607    -2.896333      0.000000
    2      4  -1.273    -2.490500      0.000000
    2      5   5.167    -0.959000      0.000000
    2      6  -0.593    -0.898000      0.000000
    2      7   3.297    -0.298714      0.000000
    2      8  -1.963    -0.506750      0.000000
    2      9  -2.343    -0.710778      0.000000
    2     10  -2.493    -0.889000      0.000000
    2     11   3.937    -0.450273      0.000000
    2     12   4.547    -0.033833      0.000000
    2     13   4.177     0.290077      0.000000
    2     14  -1.893     0.134143      0.000000
    2     15   4.757     0.442333      0.773571
    2     16  -3.893     0.587667      0.023571
    2     17  -7.083     0.397000     -0.620714
    2     18   2.897     0.483000      0.297857
    2     19   4.417     0.862333     -0.053571
    2     20  -2.973    

In [19]:
def add_rolling(df, win=WIN):
    df = df.sort_values(["unit", "cycle"]).copy()
    g = df.groupby("unit")

    # 1) التنعيم أولاً
    for c in DEV:
        df[c + "_mean"] = g[c].rolling(win, min_periods=1).mean().values

    # 2) الاشتقاق من القيمة المنعّمة، لا من الخام
    g2 = df.groupby("unit")
    for c in DEV:
        m = c + "_mean"
        df[c + "_slope"] = (
            (df[m] - g2[m].shift(win - 1)) / (win - 1)
        ).fillna(0).values

    return df


tr3 = add_rolling(tr2)
va3 = add_rolling(va2)

print(tr3[tr3.unit == 2][["cycle", "s4_dev_mean", "s4_dev_slope"]]
      .iloc[12:22].to_string(index=False))

 cycle  s4_dev_mean  s4_dev_slope
    13     0.290077      0.000000
    14     0.134143      0.000000
    15     0.442333      0.465381
    16     0.587667      0.409690
    17     0.397000      0.235238
    18     0.483000      0.212393
    19     0.862333      0.130095
    20     0.319667      0.086976
    21     0.021000      0.022837
    22    -0.177667      0.023506


In [20]:
COLS_3 = FEATURES + DEV + ROLL

m3 = LinearRegression()
m3.fit(tr3[COLS_3], tr3["RUL_capped"])

rmse3 = mean_squared_error(
    va3["RUL_capped"], m3.predict(va3[COLS_3])
) ** 0.5

print("قراءات فقط        :", round(rmse, 2))
print("+ انحرافات        :", round(rmse2, 2))
print("+ تنعيم وميل      :", round(rmse3, 2))
print("عدد الأعمدة       :", len(COLS_3))

قراءات فقط        : 19.39
+ انحرافات        : 16.09
+ تنعيم وميل      : 14.56
عدد الأعمدة       : 68


In [21]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf.fit(tr3[COLS_3], tr3["RUL_capped"])

rmse_rf = mean_squared_error(
    va3["RUL_capped"], rf.predict(va3[COLS_3])
) ** 0.5

# للمقارنة: نفس الغابة على القراءات الخام فقط
rf_raw = RandomForestRegressor(
    n_estimators=200, random_state=42, n_jobs=-1
)
rf_raw.fit(tr3[FEATURES], tr3["RUL_capped"])

rmse_rf_raw = mean_squared_error(
    va3["RUL_capped"], rf_raw.predict(va3[FEATURES])
) ** 0.5

print("خطي  + كل الأعمدة   :", round(rmse3, 2))
print("غابة + قراءات فقط   :", round(rmse_rf_raw, 2))
print("غابة + كل الأعمدة   :", round(rmse_rf, 2))

خطي  + كل الأعمدة   : 14.56
غابة + قراءات فقط   : 16.06
غابة + كل الأعمدة   : 11.04


In [22]:
rmse_tr = mean_squared_error(
    tr3["RUL_capped"], rf.predict(tr3[COLS_3])
) ** 0.5

print("RMSE تدريب :", round(rmse_tr, 2))
print("RMSE تحقق  :", round(rmse_rf, 2))
print("الفجوة     :", round(rmse_rf - rmse_tr, 2))

RMSE تدريب : 1.96
RMSE تحقق  : 11.04
الفجوة     : 9.08


In [23]:
results = []

for depth in [8, 12, 16, None]:
    for leaf in [1, 10, 30]:
        m = RandomForestRegressor(
            n_estimators=200,
            max_depth=depth,
            min_samples_leaf=leaf,
            random_state=42,
            n_jobs=-1
        )
        m.fit(tr3[COLS_3], tr3["RUL_capped"])

        e_tr = mean_squared_error(
            tr3["RUL_capped"], m.predict(tr3[COLS_3])) ** 0.5
        e_va = mean_squared_error(
            va3["RUL_capped"], m.predict(va3[COLS_3])) ** 0.5

        results.append({
            "depth": str(depth),
            "leaf": leaf,
            "train": round(e_tr, 2),
            "val": round(e_va, 2),
            "gap": round(e_va - e_tr, 2),
        })

print(pd.DataFrame(results).to_string(index=False))

depth  leaf  train   val  gap
    8     1   7.53 11.60 4.07
    8    10   7.70 11.57 3.86
    8    30   8.29 11.56 3.27
   12     1   3.86 11.15 7.29
   12    10   4.97 11.16 6.19
   12    30   7.01 11.28 4.27
   16     1   2.38 11.06 8.68
   16    10   4.41 11.10 6.69
   16    30   6.94 11.26 4.32
 None     1   1.96 11.04 9.08
 None    10   4.38 11.10 6.72
 None    30   6.94 11.26 4.32


In [24]:
# نستخدم أفضل موديل: rf (العمق الافتراضي)
va_check = va3.copy()
va_check["pred"] = rf.predict(va3[COLS_3])
va_check["err"] = va_check["pred"] - va_check["RUL_capped"]

# تقسيم الصفوف حسب الهدف الحقيقي
bins = [-1, 20, 50, 80, 119, 120]
labels = ["0-20 (قرب العطل)", "21-50", "51-80",
          "81-119", "120 (السقف)"]
va_check["band"] = pd.cut(va_check["RUL_capped"],
                          bins=bins, labels=labels)

summary = va_check.groupby("band", observed=True).apply(
    lambda g: pd.Series({
        "عدد الصفوف": len(g),
        "RMSE": (g["err"] ** 2).mean() ** 0.5,
        "متوسط الانحياز": g["err"].mean(),
    })
).round(2)

print(summary.to_string())

                  عدد الصفوف   RMSE  متوسط الانحياز
band                                               
0-20 (قرب العطل)       420.0   5.85            3.57
21-50                  600.0  11.36            5.64
51-80                  600.0  16.75            8.53
81-119                 780.0  13.92            0.63
120 (السقف)           1670.0   7.13           -3.78


C:\Users\Baraa Hereh\AppData\Local\Temp\ipykernel_19972\2078301482.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = va_check.groupby("band", observed=True).apply(


In [25]:
def nasa_score(y_true, y_pred):
    """
    دالة تسجيل NASA: تعاقب التنبؤ المتفائل أشد من المتحفظ.
    أقل = أفضل.
    """
    d = np.asarray(y_pred) - np.asarray(y_true)
    return np.where(d < 0,
                    np.exp(-d / 13) - 1,
                    np.exp(d / 10) - 1).sum()


y_va = va3["RUL_capped"].values

for name, mdl, cols in [
    ("خطي", m3, COLS_3),
    ("غابة", rf, COLS_3),
]:
    p = mdl.predict(va3[cols])
    print(f"{name:6s} | RMSE {mean_squared_error(y_va, p)**0.5:6.2f}"
          f" | NASA {nasa_score(y_va, p):10.0f}")

خطي    | RMSE  14.56 | NASA      13605
غابة   | RMSE  11.04 | NASA       8358


In [26]:
p_rf = rf.predict(va3[COLS_3])
d = p_rf - y_va

opt = d[d >= 0]    # تنبؤات متفائلة (خطرة)
cons = d[d < 0]    # تنبؤات متحفظة (آمنة)

s_opt = (np.exp(opt / 10) - 1).sum()
s_cons = (np.exp(-cons / 13) - 1).sum()

print(f"متفائلة : {len(opt):5d} صف  ({len(opt)/len(d)*100:.0f}%)"
      f"  |  مساهمة بالنتيجة: {s_opt:9.0f}")
print(f"متحفظة  : {len(cons):5d} صف  ({len(cons)/len(d)*100:.0f}%)"
      f"  |  مساهمة بالنتيجة: {s_cons:9.0f}")
print(f"\nأسوأ تنبؤ متفائل: +{opt.max():.1f} دورة")

متفائلة :  1710 صف  (42%)  |  مساهمة بالنتيجة:      6426
متحفظة  :  2360 صف  (58%)  |  مساهمة بالنتيجة:      1932

أسوأ تنبؤ متفائل: +40.4 دورة


In [27]:
va_check["d"] = p_rf - y_va

worst = va_check.nlargest(15, "d")[
    ["unit", "cycle", "RUL_capped", "pred", "d"]
].round(1)

print("أسوأ 15 تنبؤ متفائل:")
print(worst.to_string(index=False))

print("\nتوزيع الخطأ الموجب:")
print(pd.Series(opt).describe().round(1).to_string())

أسوأ 15 تنبؤ متفائل:
 unit  cycle  RUL_capped  pred    d
   77    104          50  90.4 40.4
   91     81          54  93.9 39.9
   77    100          54  93.8 39.8
   91     86          49  88.5 39.5
   13     90          73 112.2 39.2
   45     85          73 112.0 39.0
   13     93          70 108.8 38.8
   13     91          72 110.6 38.7
   77     99          55  93.6 38.6
   13     94          69 107.5 38.5
   45     87          71 109.0 38.0
   13     92          71 108.7 37.7
   45     86          72 109.6 37.6
   13     89          74 111.4 37.4
   45     88          70 107.2 37.2

توزيع الخطأ الموجب:
count    1710.0
mean       10.1
std         9.3
min         0.0
25%         2.8
50%         7.2
75%        15.0
max        40.4


In [28]:
# نجهّز الـ test بنفس المعالجة
t = test.copy()
t2 = add_deviation(t)
t3 = add_rolling(t2)

# آخر دورة من كل محرك — هي نقطة التنبؤ المطلوبة
last = t3.groupby("unit").tail(1).sort_values("unit")

pred_test = rf.predict(last[COLS_3])
truth = np.clip(rul_true["RUL"].values, 0, 120)

print("RMSE على الاختبار:", 
      round(mean_squared_error(truth, pred_test) ** 0.5, 2))

RMSE على الاختبار: 13.22


In [30]:
def predict_rul(engine_history, model=rf, cap=120):
    """
    تتنبأ بالعمر المتبقي لمحرك وتصنّف حالته.
    """
    df = engine_history.copy()

    if len(df) < 20:
        raise ValueError(
            f"يلزم 20 دورة على الأقل، المتوفر {len(df)}")

    df = df.sort_values("cycle")
    df["unit"] = 1

    df = add_deviation(df)
    df = add_rolling(df)

    rul = float(model.predict(df.tail(1)[COLS_3])[0])
    rul = max(0.0, min(rul, cap))     # ضمن المجال المنطقي

    if rul <= 20:
        status = "حرج - صيانة فورية"
    elif rul <= 50:
        status = "تحذير - جدولة صيانة"
    elif rul < cap:
        status = "مراقبة"
    else:
        status = "سليم"

    return {
        "RUL": round(rul),
        "الحالة": status,
        "دورات مسجلة": len(engine_history),
    }


# اختبار على عدة محركات
for u in [3, 16, 31, 50]:
    r = predict_rul(test[test.unit == u])
    actual = min(rul_true["RUL"].iloc[u - 1], 120)
    print(f"محرك {u:3d} | تنبؤ {r['RUL']:3d} | حقيقة {actual:3d} "
          f"| {r['الحالة']}")

محرك   3 | تنبؤ  56 | حقيقة  69 | مراقبة
محرك  16 | تنبؤ  89 | حقيقة  84 | مراقبة
محرك  31 | تنبؤ  17 | حقيقة   8 | حرج - صيانة فورية
محرك  50 | تنبؤ  98 | حقيقة  79 | مراقبة
